In [1]:
import torch
import torch.nn as nn
import math

Step by Step Rope Intuition

In [14]:
# The formula is θ = p * ωᵢ, where ωᵢ = 1 / 10000^(2i / d).
d_head = 64
theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
print(theta)

tensor([1.0000e+00, 7.4989e-01, 5.6234e-01, 4.2170e-01, 3.1623e-01, 2.3714e-01,
        1.7783e-01, 1.3335e-01, 1.0000e-01, 7.4989e-02, 5.6234e-02, 4.2170e-02,
        3.1623e-02, 2.3714e-02, 1.7783e-02, 1.3335e-02, 1.0000e-02, 7.4989e-03,
        5.6234e-03, 4.2170e-03, 3.1623e-03, 2.3714e-03, 1.7783e-03, 1.3335e-03,
        1.0000e-03, 7.4989e-04, 5.6234e-04, 4.2170e-04, 3.1623e-04, 2.3714e-04,
        1.7783e-04, 1.3335e-04])


In [15]:
max_seq_len = 8
positions = torch.arange(max_seq_len).unsqueeze(1)
print(positions)

tensor([[0],
        [1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7]])


In [16]:
freqs = positions * theta.unsqueeze(0)
print(freqs)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.0000e+00, 7.4989e-01, 5.6234e-01, 4.2170e-01, 3.1623e-01, 2.3714e-01,
         1.7783e-01, 1.3335e-01, 1.0000e-01, 7.4989e-02, 5.6234e-02, 4.2170e-02,
         3.1623e-02, 2.3714e-02, 1.7783e-02, 1.3335e-02, 1.0000e-02, 7.4989e-03,
         5.6234e-03, 4.2170e-03, 3.1623e-03, 2.3714e-03, 1.7783e-03, 1.3335e-03,
         1.0000e-03, 7.4989e-04, 5.6234e-04, 4.2170e-04, 3.1623e-04, 2.3714e-04,
         1.7783e-04, 1.3335e-04],
        [2.0000e+00, 1.4998e+00, 1.1247e+00, 8.4339e-01, 6.3246e-01, 4.7427e-01,
         3.5566e-01, 2.6670e-01, 2.0000e-

In [17]:
freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
print(freqs_cis)

tensor([[ 1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j,  1.0000+0.0000e+00j],
        [ 0.5403+8.4147e-01j,  0.7318+6.8156e-01j,  0.8460+5.3317e-01j,
          0.9124+4.0931e-01j,  0.9504+3.1098e-01j,  0.9720+2.3492e-01j,
          0.9842+1.7689e-01j,  0.9911+1.3296e-01j,  0.9950+9.9833e-02j,
          0.

In [24]:
import torch

# Define hyperparameter dimensions
batch_size = 2
num_heads = 4
seq_len = 8
d_head = 64  # Must be an even number because RoPE pairs dimensions into 2s

# 1. Create dummy input tensor 'x'
# Shape: (batch, num_heads, seq_len, d_head) -> (2, 4, 8, 64)
x = torch.randn(batch_size, num_heads, seq_len, d_head)

print("Original x shape:", x.shape)

# 2. Reshape to group dimensions into 2D pairs
# Shape becomes: (2, 4, 8, 32, 2)
x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)

print("After reshape (paired in 2s):", x_reshaped.shape)

# 3. View as complex numbers
# Shape becomes: (2, 4, 8, 32) where each value is a complex number (a + ib)
x_complex = torch.view_as_complex(x_reshaped)

print("After view_as_complex:", x_complex.shape)
print("Complex tensor dtype:", x_complex.dtype)

# --- Let's inspect the first pair of the first token! ---
print("\n--- Inspecting First Pair ---")
print("Original real values:", x[0, 0, 0, :2])  # First 2 real numbers
print("Complex value:", x_complex[0, 0, 0, 0])  # Represented as a + ib

Original x shape: torch.Size([2, 4, 8, 64])
After reshape (paired in 2s): torch.Size([2, 4, 8, 32, 2])
After view_as_complex: torch.Size([2, 4, 8, 32])
Complex tensor dtype: torch.complex64

--- Inspecting First Pair ---
Original real values: tensor([1.3264, 0.2323])
Complex value: tensor(1.3264+0.2323j)


In [19]:
x_freqs_cis = freqs_cis[:seq_len, :].unsqueeze(0).unsqueeze(0)
print(x_freqs_cis)

tensor([[[[ 1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
            1.0000+0.0000e+00j,  1.0000+0.0000e+00j],
          [ 0.5403+8.4147e-01j,  0.7318+6.8156e-01j,  0.8460+5.3317e-01j,
            0.9124+4.0931e-01j,  0.9504+3.1098e-01j,  0.9720+2.3492e-01j,
            0.9842+1.7689e-01j,  0.9911+1.3296e-01j,  0.99

In [ ]:
x_rotated = x_complex * x_freqs_cis
# Convert back to real number representation
x_rotated = torch.view_as_real(x_rotated)
# Reshape back to the original d_head dimension
x_rotated = x_rotated.flatten(3)
        
return x_rotated.type_as(x)


Final Complete Code

#
# LISTING 3.2: Building the Fused MLA and Decoupled RoPE Module
#
import torch
import torch.nn as nn
import math

class RotaryPositionalEncoding(nn.Module):
    """
    Helper module to apply Rotary Positional Encoding (RoPE).
    This is not added to the embeddings but is applied directly to
    the Query and Key vectors.
    """
    def __init__(self, d_head, max_seq_len=2048):
        super().__init__()
        # Precompute the theta values for the rotational matrix
        theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('theta', theta)
        
        # Precompute the frequency terms (m * theta) for all positions
        positions = torch.arange(max_seq_len).unsqueeze(1)
        freqs = positions * self.theta.unsqueeze(0)
        
        # Create the complex number representation for rotation
        # The real part is cos(freqs) and the imaginary part is sin(freqs)
        self.register_buffer('freqs_cis', torch.polar(torch.ones_like(freqs), freqs))

    def forward(self, x):
        # x shape: (batch, num_heads, seq_len, d_head)
        seq_len = x.shape[2]
        
        # Reshape x to treat pairs of dimensions as complex numbers
        x_complex = x.float().reshape(*x.shape[:-1], -1, 2)
        # Convert to PyTorch complex type
        x_complex = torch.view_as_complex(x_complex)
        
        # Get the precomputed frequencies for the current sequence length
        freqs_cis = self.freqs_cis[:seq_len, :].unsqueeze(0).unsqueeze(0)
        
        # Apply rotation by multiplying in the complex domain
        # This rotates each pair of dimensions by the angle m * theta_i
        x_rotated = x_complex * freqs_cis
        
        # Convert back to real number representation
        x_rotated = torch.view_as_real(x_rotated)
        # Reshape back to the original d_head dimension
        x_rotated = x_rotated.flatten(3)
        
        return x_rotated.type_as(x)


class DeepSeekAttention(nn.Module):
    """
    The full, state-of-the-art attention mechanism from DeepSeek, combining
    Multi-Head Latent Attention (MLA) with Decoupled Rotary Positional
    Encoding (RoPE).
    """
    def __init__(self, d_model, num_heads, d_latent, d_rope, dropout=0.0, max_seq_len=2048):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent
        self.d_rope = d_rope # Dimension for positional vectors
        
        # --- A: Content Path (Pure MLA) ---
        self.W_q_content = nn.Linear(d_model, d_model)
        self.W_dkv_content = nn.Linear(d_model, d_latent)
        self.W_uk_content = nn.Linear(d_latent, d_model)
        self.W_uv_content = nn.Linear(d_latent, d_model)
        
        # --- B: Position Path (RoPE Applied) ---
        self.W_k_pos = nn.Linear(d_model, d_rope * num_heads)
        self.W_q_pos = nn.Linear(d_model, d_rope * num_heads)
        
        # RoPE module to apply the rotations
        self.rope = RotaryPositionalEncoding(d_rope, max_seq_len)
        
        # --- C: Final Output Projection ---
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(
            torch.ones(1, 1, max_seq_len, max_seq_len), diagonal=1).bool())

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # --- A: Content Path Calculation ---
        # This path is cache-friendly and position-agnostic.
        q_c = self.W_q_content(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        c_kv = self.W_dkv_content(x) # This is what gets cached for the content path.
        k_c = self.W_uk_content(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        v_c = self.W_uv_content(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        
        # --- B: Position Path Calculation ---
        # This path handles the positional information.
        q_r_unrotated = self.W_q_pos(x).view(batch_size, seq_len, self.num_heads, self.d_rope).transpose(1, 2)
        k_r_unrotated = self.W_k_pos(x).view(batch_size, seq_len, self.num_heads, self.d_rope).transpose(1, 2)

        # Apply RoPE to the positional Query and Key vectors
        q_r = self.rope(q_r_unrotated)
        k_r = self.rope(k_r_unrotated) # This is what gets cached for the position path.
        
        # --- C: Combining Paths for Final Attention Score ---
        # The final score is the sum of content and position scores.
        content_scores = (q_c @ k_c.transpose(-2, -1)) / (self.d_head ** 0.5)
        position_scores = (q_r @ k_r.transpose(-2, -1)) / (self.d_rope ** 0.5)
        
        attn_scores = content_scores + position_scores
        
        # --- D: Final Steps (Masking, Softmax, Output) ---
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len], float('-inf'))
            
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # The final context vector is computed using only the content value matrix (v_c)
        context_vector = (attn_weights @ v_c).transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model)
            
        output = self.W_o(context_vector)
        return output

# --- Usage Example ---
d_model = 512
num_heads = 8
d_latent = 128
d_rope = 64 # Dimension for RoPE, typically d_head or smaller
batch_size = 4
seq_len = 64

# Instantiate the full attention layer
deepseek_attn_layer = DeepSeekAttention(d_model, num_heads, d_latent, d_rope)

# Create a dummy input tensor
dummy_input = torch.randn(batch_size, seq_len, d_model)

# Pass the input through the layer
output = deepseek_attn_layer(dummy_input)

print("DeepSeekAttention Layer successful!")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")